In [15]:
import numpy as np
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import time

print(f"GPU: {cuda.Device(0).name()}")
print(f"CUDA version: {cuda.get_version()}")
print("=" * 70)

def vector_sum_cpu(vector):
    """Vector sum on CPU with sequential loop"""
    total = 0
    for elem in vector:
        total += elem
    return total

def vector_sum_gpu(vector):
    """Vector sum on GPU with parallel reduction"""
    N = len(vector)
    
    kernel_code = """
    __global__ void sumKernel(int* result, int* a, unsigned int size) {
        __shared__ int sharedSum[256];
        
        int index = blockIdx.x * blockDim.x + threadIdx.x;
        int localSum = 0;
        
        if (index < size) {
            localSum = a[index];
        }
        
        sharedSum[threadIdx.x] = localSum;
        __syncthreads();
        
        for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
            if (threadIdx.x < stride) {
                sharedSum[threadIdx.x] += sharedSum[threadIdx.x + stride];
            }
            __syncthreads();
        }
        
        if (threadIdx.x == 0) {
            atomicAdd(result, sharedSum[0]);
        }
    }
    """
    
    mod = SourceModule(kernel_code)
    sum_kernel = mod.get_function("sumKernel")
    
    vector_gpu = cuda.mem_alloc(vector.nbytes)
    result_gpu = cuda.mem_alloc(4)  
    
    initial_value = np.array([0], dtype=np.int32)
    cuda.memcpy_htod(result_gpu, initial_value)
    
    cuda.memcpy_htod(vector_gpu, vector)
    
    block_size = 256
    grid_size = (N + block_size - 1) // block_size
    
    start = time.perf_counter()
    
    sum_kernel(result_gpu, vector_gpu, np.int32(N), 
               block=(block_size, 1, 1), grid=(grid_size, 1))
    cuda.Context.synchronize()
    
    end = time.perf_counter()
    total_time = end - start
    
    result = np.empty(1, dtype=np.int32)
    cuda.memcpy_dtoh(result, result_gpu)
    
    vector_gpu.free()
    result_gpu.free()
    
    return result[0], total_time

test_sizes = [1000, 10000, 100000, 500000, 1000000]

print("\nRunning tests...")
print("=" * 70)

results = []

for size in test_sizes:
    print(f"\nVector size: {size} elements")
    print("-" * 50)
    

    vector = np.random.randint(1, 100, size=size, dtype=np.int32)
    

    print("CPU time...", end=" ", flush=True)
    cpu_start = time.perf_counter()
    cpu_result = vector_sum_cpu(vector)
    cpu_end = time.perf_counter()
    cpu_time = cpu_end - cpu_start
    print(f"{cpu_time:.6f} sec")
    print(f"CPU sum: {cpu_result}")
    
    
    print("GPU time...", end=" ", flush=True)
    gpu_result, gpu_time = vector_sum_gpu(vector)
    print(f"{gpu_time:.6f} sec")
    print(f"GPU sum: {gpu_result}")
    
    
    if cpu_time < gpu_time:
        speedup = gpu_time / cpu_time
        speedup_str = f"CPU is {speedup:.2f}x faster"
        speedup_value = -speedup
    else:
        speedup = cpu_time / gpu_time
        speedup_str = f"{speedup:.2f}x"
        speedup_value = speedup
    
    print(f"Speedup: {speedup_str}")
    
    results.append({
        'size': size,
        'cpu_time': cpu_time,
        'gpu_time': gpu_time,
        'speedup': speedup_value,
        'cpu_result': cpu_result,
        'gpu_result': gpu_result
    })

print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"{'Size':<12} {'CPU(s)':<12} {'GPU(s)':<12} {'Speedup':<15}")
print("-" * 55)

for r in results:
    if r['speedup'] < 0:
        speedup_str = f"CPU {-r['speedup']:.2f}x"
    else:
        speedup_str = f"{r['speedup']:.2f}x"
    print(f"{r['size']:<12} {r['cpu_time']:<12.6f} {r['gpu_time']:<12.6f} {speedup_str:<15}")


GPU: NVIDIA GeForce RTX 5070
CUDA version: (12, 9, 0)

Running tests...

Vector size: 1000 elements
--------------------------------------------------
CPU time... 0.000051 sec
CPU sum: 49995
GPU time... 0.000056 sec
GPU sum: 49995
Speedup: CPU is 1.09x faster

Vector size: 10000 elements
--------------------------------------------------
CPU time... 0.000445 sec
CPU sum: 500152
GPU time... 0.000069 sec
GPU sum: 500152
Speedup: 6.49x

Vector size: 100000 elements
--------------------------------------------------
CPU time... 0.004581 sec
CPU sum: 4996498
GPU time... 0.000042 sec
GPU sum: 4996498
Speedup: 108.29x

Vector size: 500000 elements
--------------------------------------------------
CPU time... 0.022446 sec
CPU sum: 25003161
GPU time... 0.000102 sec
GPU sum: 25003161
Speedup: 220.49x

Vector size: 1000000 elements
--------------------------------------------------
CPU time... 0.047020 sec
CPU sum: 49967571
GPU time... 0.000069 sec
GPU sum: 49967571
Speedup: 680.46x

RESULTS SUM